In [ ]:
# Sistema: operación de transporte. Pregunta: ¿qué conductores tienen mayor actividad y qué semanas están bajo su promedio?
#
# lectura ──► mapper (driverId, medidas) ──► reducer ──► escritura de resultados

import csv
from collections import defaultdict
from pathlib import Path

ROOT = next(
    p
    for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "data").is_dir() and (p / "submission").is_dir()
)


def load(name):
    with (ROOT / "data" / name).open(encoding="utf-8", newline="") as file:
        return list(csv.DictReader(file))


timesheet, drivers = load("timesheet.csv"), load("drivers.csv")


In [ ]:
# SQL:
# SELECT
#     driverId,
#     SUM("hours-logged") AS total_hours,
#     SUM("miles-logged") AS total_miles,
#     AVG("hours-logged") AS mean_hours,
#     MIN("hours-logged") AS min_hours,
#     MAX("hours-logged") AS max_hours
# FROM timesheet
# GROUP BY driverId;
# MAPPER: cada registro semanal produce medidas parciales con driverId como clave.

partials = defaultdict(lambda: [0.0, 0.0, 0, float("inf"), float("-inf")])
for row in timesheet:
    hours, miles = float(row["hours-logged"]), float(row["miles-logged"])
    value = partials[row["driverId"]]
    value[0] += hours
    value[1] += miles
    value[2] += 1
    value[3] = min(value[3], hours)
    value[4] = max(value[4], hours)
aggregates = {
    key: {
        "hours-logged": round(v[0], 2),
        "miles-logged": round(v[1], 2),
        "mean_hours-logged": round(v[0] / v[2], 2),
        "min_hours-logged": round(v[3], 2),
        "max_hours-logged": round(v[4], 2),
    }
    for key, v in partials.items()
}


In [ ]:
# SQL:
# SELECT t.*
# FROM timesheet AS t
# JOIN summary AS s USING (driverId)
# WHERE t."hours-logged" < s."mean_hours-logged";
# REDUCER: consolida cada driverId; la segunda pasada compara cada semana con el promedio reducido.

names = {row["driverId"]: row["name"] for row in drivers}
summary = [
    {"driverId": key, "name": names[key], **value} for key, value in aggregates.items()
]
summary.sort(key=lambda row: int(row["driverId"]))
below_average = [
    {**row, "mean_hours-logged": aggregates[row["driverId"]]["mean_hours-logged"]}
    for row in timesheet
    if float(row["hours-logged"]) < aggregates[row["driverId"]]["mean_hours-logged"]
]
# SQL:
# SELECT *
# FROM summary
# ORDER BY "miles-logged" DESC
# LIMIT 10;
top10 = sorted(summary, key=lambda row: row["miles-logged"], reverse=True)[:10]


In [ ]:
# ESCRITURA: los CSV conservan el resumen, las semanas bajo el promedio y los diez conductores con más millas.

for name, result in {
    "summary.csv": summary,
    "below_average_hours.csv": below_average,
    "top10_drivers.csv": top10,
}.items():
    with (ROOT / "submission" / name).open("w", encoding="utf-8", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=list(result[0]), lineterminator="\n")
        writer.writeheader()
        writer.writerows(result)
